In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

In [ ]:
# Load the dataset
data = pd.read_excel('Concrete_Data.xls')
print(data.head())

In [ ]:
# Preprocess the data
X = data.drop('Concrete compressive strength(MPa, megapascals) ', axis=1).values
y = data['Concrete compressive strength(MPa, megapascals) '].values

#-----Handle missing values if present-----
# Separate columns into two groups of continuous and integer types
continuous_cols = [col for col in data.columns if data[col].dtype == 'float64']
integer_cols = [col for col in data.columns if data[col].dtype == 'int64']
# [Citation] Above two lines to separate columns into two groups are generated by Colab code autocompletion.

# [Citation] The codes imputing missing values are generated by Gemini in Colab.
# Impute missing values in columns of continuous type with the mean
imputer_continuous = SimpleImputer(strategy='mean')
X = imputer_continuous.fit_transform(X)

# Impute missing values in columns of integer type with the most frequent
imputer_integer = SimpleImputer(strategy='most_frequent')
X = imputer_integer.fit_transform(X)

#-----Standardize features-----
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

#-----Split the data-----
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

# Set y_test on random_state=42 as ground_truth
# This variable will be used in bias-variance analysis parts.
ground_truth_test = y_test

In [ ]:
# (a) Polynomial Regression
from sklearn.preprocessing import PolynomialFeatures

def polynomial(X_train, X_test, y_train, y_test, degree):
    #-----Blank: Transform features into polynomial features----
    # Create a PolynomialFeatures object with the given degree (include bias term)
    # Transform both the training and test data

    # TODO: Fill in the code below
    poly = PolynomialFeatures(degree, include_bias=True)
    X_train_poly = poly.fit_transform(X_train)
    X_test_poly = poly.transform(X_test)
    #--------------------------------------------------------------

    #-----Blank: Fit linear regression model and evaluate performance----
    # TODO:
    #-Initialize and train LinearRegression model
    #-Predict on both training and test sets
    #-Compute train and test MSE
    #-Append errors to respective lists
    model_poly = LinearRegression()
    model_poly.fit(X_train_poly, y_train)

    y_train_pred = model_poly.predict(X_train_poly)
    y_test_pred = model_poly.predict(X_test_poly)

    train_mse = mean_squared_error(y_train, y_train_pred)
    test_mse = mean_squared_error(y_test, y_test_pred)

    return y_train_pred, y_test_pred, train_mse, test_mse
    #----------------------------------------------------------------------

In [ ]:
degrees = [1, 2, 3, 4, 5, 7]
poly_train_errors = []
poly_test_errors = []

for degree in degrees:
    y_train_pred, y_test_pred, train_mse, test_mse = polynomial(X_train, X_test, y_train, y_test, degree)

    poly_train_errors.append(train_mse)
    poly_test_errors.append(test_mse)

In [ ]:
# (b) Gaussian Basis Regression

# [Citation] More than half of the structures of two functions below, 'create_gaussian_design_matrix'
# and 'create_full_gaussian_design_matrix', are generated by ChatGPT----------
def create_gaussian_design_matrix(x, degree, sigma='auto', include_bias=True):
    """
    Create a design matrix using Gaussian basis functions.
    Args:
    x: numpy array of shape (n_samples,)
    degree: number of Gaussian basis functions
    sigma: width of the Gaussian functions
    Returns:
    X: design matrix with shape (n_samples, degree) or (n_samples, degree + 1) including bias term
    """
    #-----Blank: Fill in Gaussian basis transformation----
    # TODO: Fill in the loop to compute Gaussian basis functions
    n_samples = x.shape[0]
    bias_term = 1 if include_bias else 0
    X = np.ones((n_samples, degree + bias_term))

    if degree == 1:
        mus = [np.mean(x)]
        sigma = 1 if sigma == 'auto' else sigma
    else:
        mus = np.linspace(np.min(x), np.max(x), degree)
        sigma = mus[1] - mus[0] if sigma == 'auto' else sigma
        sigma = max(sigma, 1e-6)

    for j in range(degree):
        mu = mus[j]
        X[:, j + bias_term] = np.exp(- ((x - mu) ** 2) / (2 * sigma**2))
    #--------------------------------------------------------
    return X

#-----Create whole columns of basis for all features-----
def create_full_gaussian_design_matrix(X, degree, sigma='auto'):
    n_features = X.shape[1]
    basis_list = []
    for i in range(n_features):
        x_feature = X[:, i]  # i-th feature (1D vector)
        include_bias = True if i == 0 else False
        X_feature_basis = create_gaussian_design_matrix(x_feature, degree, sigma, include_bias)
        basis_list.append(X_feature_basis)
    X_full_basis = np.concatenate(basis_list, axis=1)  # (N, D*degree+1)
    return X_full_basis

def gaussian(X_train, X_test, y_train, y_test, degree, sigma='auto'):
    X_train_gauss = create_full_gaussian_design_matrix(X_train, degree, sigma='auto')
    X_test_gauss = create_full_gaussian_design_matrix(X_test, degree, sigma='auto')

    #---Blank: Fit Gaussian basis regression model and evaluate performance--
    # TODO:
    #-Initialize and train LinearRegression model
    #-Predict on both training and test sets
    #-Compute train and test MSE
    #-Append errors to respective lists
    model_gauss = LinearRegression()
    model_gauss.fit(X_train_gauss, y_train)

    y_train_pred = model_gauss.predict(X_train_gauss)
    y_test_pred = model_gauss.predict(X_test_gauss)

    train_mse = mean_squared_error(y_train, y_train_pred)
    test_mse = mean_squared_error(y_test, y_test_pred)

    return y_train_pred, y_test_pred, train_mse, test_mse
    #----------------------------------------------------------------------

In [ ]:
gaussian_degrees = [1, 2, 3, 4, 5, 7, 10]
gauss_train_errors = []
gauss_test_errors = []

for degree in gaussian_degrees:
    y_train_pred, y_test_pred, train_mse, test_mse = gaussian(X_train, X_test, y_train, y_test, degree, sigma='auto')

    gauss_train_errors.append(train_mse)
    gauss_test_errors.append(test_mse)

In [ ]:
plt.plot(degrees, poly_train_errors, '--', c='orange', label='Train MSE')
plt.plot(degrees, poly_test_errors, 'r-', label='Test MSE')
plt.xlabel('Degree of Polynomial Regression')
plt.ylabel('Mean Squared Error (log scale)')
plt.yscale('log')
plt.legend()
plt.title('Polynomial Regression Error')
plt.grid(True)
plt.show()

In [ ]:
idx_min_train = np.argmin(poly_train_errors)
idx_min_test = np.argmin(poly_test_errors)
print(f'Polynomial Regression Train MSE is the smallest, {poly_train_errors[idx_min_train]}, at degree = {degrees[idx_min_train]}')
print(f'Polynomial Regression Test MSE is the smallest, {poly_test_errors[idx_min_test]}, at degree = {degrees[idx_min_test]}')

In [ ]:
plt.plot(gaussian_degrees, gauss_train_errors, 'c--', label='Train MSE')
plt.plot(gaussian_degrees, gauss_test_errors, 'b-', label='Test MSE')
plt.xlabel('Degree of Gaussian Basis')
plt.ylabel('Mean Squared Error (log scale)')
plt.yscale('log')
plt.legend()
plt.title('Gaussian Basis Regression Error')
plt.grid(True)
plt.show()

In [ ]:
idx_min_train = np.argmin(gauss_train_errors)
idx_min_test = np.argmin(gauss_test_errors)
print(f'Gaussian Basis Train MSE is the smallest, {gauss_train_errors[idx_min_train]}, at degree = {gaussian_degrees[idx_min_train]}')
print(f'Gaussian Basis Test MSE is the smallest, {gauss_test_errors[idx_min_test]}, when degree = {gaussian_degrees[idx_min_test]}')

In [ ]:
# Plot comparison of test errors
plt.figure(figsize=(10,6))
plt.plot(degrees, poly_train_errors, '--', c='orange', label='Polynomial Regression Train MSE')
plt.plot(degrees, poly_test_errors, 'r-', label='Polynomial Regression Test MSE')
plt.plot(gaussian_degrees, gauss_train_errors, 'c--', label='Gaussian Basis Train MSE')
plt.plot(gaussian_degrees, gauss_test_errors, 'b-', label='Gaussian Basis Test MSE')
plt.xlabel('Model Complexity (Degree or Number of Basis Functions)')
plt.ylabel('Mean Squared Error (log scale)')
plt.yscale('log')
plt.title('Comparison of Basis Functions on Concrete Data')
plt.legend()
plt.show()

In [ ]:
from numpy.linalg import norm

np.random.seed(0)

n_models = 100

poly_squared_bias_list = []
poly_variance_list = []

# [Citation] The codes in the loop "for i in range(n_models)" are made from snippet provided by ChatGPT.
for degree in degrees:
    predictions = []  # To store predictions from each model
    # Train n_models models on different training samples
    for i in range(n_models):
        # Generate a new training set for each model
        X_train_var, _, y_train_var, _ = train_test_split(X_scaled, y, test_size=0.2, random_state=1000+i)
        y_train_var += np.random.normal(0, 0.1*np.std(y_train_var), len(y_train_var))
        y_train_pred, y_test_pred, train_mse, test_mse = polynomial(X_train_var, X_test, y_train_var, y_test, degree)

        predictions.append(y_test_pred)

    predictions = np.array(predictions)  # shape: (n_models, n_test_samples)
    # Compute the average prediction for each test sample
    avg_prediction = np.mean(predictions, axis=0)
    # Compute integrated squared bias (average squared difference between avg_prediction and ground truth)
    squared_bias = norm(avg_prediction - ground_truth_test) ** 2 / len(y_test)
    # Compute integrated variance (average variance across test samples)
    variance = np.mean(np.var(predictions, axis=0))

    poly_squared_bias_list.append(squared_bias)
    poly_variance_list.append(variance)

plt.figure()
plt.plot(degrees, poly_squared_bias_list, label='(Bias)^2')
plt.plot(degrees, poly_variance_list, label='Variance')
plt.plot(degrees, np.array(poly_squared_bias_list) + np.array(poly_variance_list), label='Total Error')
plt.xlabel('Degree of Polynomial Regression')
plt.ylabel('Error (log scale)')
plt.yscale('log')
plt.legend()
plt.title('Polynomial Regression Error')
plt.grid(True)
plt.show()

In [ ]:
gauss_squared_bias_list = []
gauss_variance_list = []

for degree in gaussian_degrees:
    predictions = []  # To store predictions from each model
    # Train n_models models on different training samples
    for i in range(n_models):
        # Generate a new training set for each model
        X_train_var, _, y_train_var, _ = train_test_split(X_scaled, y, test_size=0.2, random_state=1000+i)
        y_train_var += np.random.normal(0, 0.1*np.std(y_train_var), len(y_train_var))
        y_train_pred, y_test_pred, train_mse, test_mse = gaussian(X_train_var, X_test, y_train_var, y_test, degree)

        predictions.append(y_test_pred)

    predictions = np.array(predictions)  # shape: (n_models, n_test_samples)
    # Compute the average prediction for each test sample
    avg_prediction = np.mean(predictions, axis=0)
    # Compute integrated squared bias (average squared difference between avg_prediction and ground truth)
    squared_bias = norm(avg_prediction - ground_truth_test) ** 2 / len(y_test)
    # Compute integrated variance (average variance across test samples)
    variance = np.mean(np.var(predictions, axis=0))

    gauss_squared_bias_list.append(squared_bias)
    gauss_variance_list.append(variance)

plt.figure()
plt.plot(gaussian_degrees, gauss_squared_bias_list, label='(Bias)^2')
plt.plot(gaussian_degrees, gauss_variance_list, label='Variance')
plt.plot(gaussian_degrees, np.array(gauss_squared_bias_list) + np.array(gauss_variance_list), label='Total Error')
plt.xlabel('Degree of Gaussian Basis Regression')
plt.ylabel('Error (log scale)')
plt.yscale('log')
plt.legend()
plt.title('Gaussian Basis Regression Error')
plt.grid(True)
plt.show()

In [ ]:
from sklearn.linear_model import Ridge

# Sigmoid Basis Regression with L2 regularization
def create_sigmoid_design_matrix(x, degree, sigma='auto', include_bias=True):
    """
    Create a design matrix using Sigmoid basis functions.
    Args:
    x: numpy array of shape (n_samples,)
    degree: number of Sigmoid basis functions
    sigma: width of the Sigmoid functions
    Returns:
    X: design matrix with shape (n_samples, degree) or (n_samples, degree + 1) including bias term
    """
    n_samples = x.shape[0]
    bias_term = 1 if include_bias else 0
    X = np.ones((n_samples, degree + bias_term))

    if degree == 1:
        mus = [np.mean(x)]
        sigma = 1 if sigma == 'auto' else sigma
    else:
        mus = np.linspace(np.min(x), np.max(x), degree)
        sigma = mus[1] - mus[0] if sigma == 'auto' else sigma
        sigma = max(sigma, 1e-6)

    for j in range(degree):
        mu = mus[j]
        X[:, j + bias_term] = 1 / (1 + np.exp(- (x - mu) / sigma))
    return X

def create_full_sigmoid_design_matrix(X, degree, sigma='auto'):
    n_features = X.shape[1]
    basis_list = []
    for i in range(n_features):
        x_feature = X[:, i]
        include_bias = True if i == 0 else False
        X_feature_basis = create_sigmoid_design_matrix(x_feature, degree, sigma, include_bias)
        basis_list.append(X_feature_basis)
    X_full_basis = np.concatenate(basis_list, axis=1)  # (N, D*degree+1)
    return X_full_basis

def sigmoid(X_train, X_test, y_train, y_test, degree, sigma='auto'):
    X_train_sigmoid = create_full_sigmoid_design_matrix(X_train, degree, sigma='auto')
    X_test_sigmoid = create_full_sigmoid_design_matrix(X_test, degree, sigma='auto')

    #---Blank: Fit Gaussian basis regression model and evaluate performance--
    # TODO:
    #-Initialize and train LinearRegression model
    #-Predict on both training and test sets
    #-Compute train and test MSE
    #-Append errors to respective lists
    model_sigmoid = Ridge(alpha=1e-6)
    model_sigmoid.fit(X_train_sigmoid, y_train)

    y_train_pred = model_sigmoid.predict(X_train_sigmoid)
    y_test_pred = model_sigmoid.predict(X_test_sigmoid)

    train_mse = mean_squared_error(y_train, y_train_pred)
    test_mse = mean_squared_error(y_test, y_test_pred)

    return y_train_pred, y_test_pred, train_mse, test_mse
    #----------------------------------------------------------------------

In [ ]:
sigmoid_degrees = [1, 2, 3, 4, 5, 7, 10]
sigmoid_train_errors = []
sigmoid_test_errors = []

for degree in gaussian_degrees:
    y_train_pred, y_test_pred, train_mse, test_mse = sigmoid(X_train, X_test, y_train, y_test, degree, sigma='auto')

    sigmoid_train_errors.append(train_mse)
    sigmoid_test_errors.append(test_mse)

In [ ]:
idx_min_train = np.argmin(sigmoid_train_errors)
idx_min_test = np.argmin(sigmoid_test_errors)
print(f'Sigmoid Basis Train MSE is the smallest, {sigmoid_train_errors[idx_min_train]}, at degree = {sigmoid_degrees[idx_min_train]}')
print(f'Sigmoid Basis Test MSE is the smallest, {sigmoid_test_errors[idx_min_test]}, at degree = {sigmoid_degrees[idx_min_test]}')

In [ ]:
plt.plot(sigmoid_degrees, sigmoid_train_errors, '--', c='limegreen', label='Train MSE')
plt.plot(sigmoid_degrees, sigmoid_test_errors, 'g-', label='Test MSE')
plt.xlabel('Degree of Sigmoid Basis')
plt.ylabel('Mean Squared Error')
plt.legend()
plt.title('Sigmoid Basis Regression Error')
plt.grid(True)
plt.show()

In [ ]:
# Plot comparison of test errors
plt.figure(figsize=(10,6))
plt.plot(degrees, poly_train_errors, '--', c='orange', label='Polynomial Regression Train MSE')
plt.plot(degrees, poly_test_errors, 'r-', label='Polynomial Regression Test MSE')
plt.plot(gaussian_degrees, gauss_train_errors, 'c--', label='Gaussian Basis Train MSE')
plt.plot(gaussian_degrees, gauss_test_errors, 'b-', label='Gaussian Basis Test MSE')
plt.plot(sigmoid_degrees, sigmoid_train_errors, '--', c='limegreen', label='Sigmoid Basis Train MSE')
plt.plot(sigmoid_degrees, sigmoid_test_errors, 'g-', label='Sigmoid Basis Test MSE')
plt.xlabel('Model Complexity (Degree or Number of Basis Functions)')
plt.ylabel('Mean Squared Error (log scale)')
plt.yscale('log')
plt.title('Comparison of Basis Functions on Concrete Data')
plt.legend()
plt.show()

In [ ]:
sigmoid_squared_bias_list = []
sigmoid_variance_list = []

for degree in sigmoid_degrees:
    predictions = []  # To store predictions from each model
    # Train n_models models on different training samples
    for i in range(n_models):
        # Generate a new training set for each model (noise fixed to 1)
        X_train_var, _, y_train_var, _ = train_test_split(X_scaled, y, test_size=0.2, random_state=1000+i)
        y_train_var += np.random.normal(0, 0.1*np.std(y_train_var), len(y_train_var))
        y_train_pred, y_test_pred, train_mse, test_mse = sigmoid(X_train_var, X_test, y_train_var, y_test, degree)

        predictions.append(y_test_pred)

    predictions = np.array(predictions)  # shape: (n_models, n_test_samples)
    # Compute the average prediction for each test sample
    avg_prediction = np.mean(predictions, axis=0)
    # Compute integrated squared bias (average squared difference between avg_prediction and ground truth)
    squared_bias = norm(avg_prediction - ground_truth_test) ** 2 / len(y_test)
    # Compute integrated variance (average variance across test samples)
    variance = np.mean(np.var(predictions, axis=0))

    sigmoid_squared_bias_list.append(squared_bias)
    sigmoid_variance_list.append(variance)

plt.figure()
plt.plot(sigmoid_degrees, sigmoid_squared_bias_list, label='(Bias)^2')
plt.plot(sigmoid_degrees, sigmoid_variance_list, label='Variance')
plt.plot(sigmoid_degrees, np.array(sigmoid_squared_bias_list) + np.array(sigmoid_variance_list), label='Total Error')
plt.xlabel('Degree of Sigmoid Basis Regression')
plt.ylabel('Error')
plt.legend()
plt.title('Sigmoid Basis Regression Error')
plt.grid(True)
plt.show()